# 🕹️ <span style=color:dodgerblue>LLM for video game knowledge assistance<span>

This notebook is used for explanation purpose. Once the `docker compose up` command
is executed the entire application is ready to be used.

## 📔 <span style=color:gold>What is this notebook about?</span>
This notebook is intended to explain all the pipeline and provide an overview 
of the processes involved in the application (ingestion, monitoring, evaluation, etc).  
It's a complement to the [README.md](README.md) that shows the internal mechanisms
of the scripts.

In [1]:
import time

from llm import RAGClient
from opensearchpy import OpenSearch


from IPython.core.display import Markdown

## <span style=color:green>📂 Opensearch client creation</span>

We create an opensearch client. It is used for indexation and search 
(lexical, semantic, hybrid).  
The `RAGClient` simplifies the use of the RAG.

In [2]:
# Opensearch client connection to the running docker container.
opensearch_client = OpenSearch(
    hosts=[{"host": "localhost", "port": 9200}],
    http_auth=("admin", "Opensearch16admin#"),
    use_ssl=False,
    verify_certs=False,
    ssl_show_warn=False,
)

# This client has a large set of functionalities that ease the use of the llm.
rag_client = RAGClient(
    opensearch_client,
    model="gemini-3.1-flash-lite",  # This is usually the best free model.
)

## <span style=color:lightsalmon>⛁ Ingestion</span>

This part showcases the ingestion process and how It works.  
To avoid the long waiting time this part is entirely optional and is not a pre-requisite to advance
to the other blocks of this notebook.  
It's just used for explanation purpose.

Before anything we call the helper function `setup_embedder` this will set the 
ML model for our vector/semantic search (and hybrid search).  
Note that the `setup_embedder` will return the `model_id` that was already set up when we called
`docker compose up`.

In [3]:
# Helper for downloading and preparing the ML embedder (convert text to vector).
from opensearch_utils import setup_embedder

In [4]:
# This will give us the model id for future use.
model_id = setup_embedder(opensearch_client)

Found existing model ID 'DiQCxZ8BvP1qKdxMCmks' in state: DEPLOYED
✅ Model 'DiQCxZ8BvP1qKdxMCmks' is active and deployed. Reusing it.


### 🎮 <span style=color:darkorchid>IGDB ingestion</span>

**You can skip this section if you want.**

> <span style=color:gold>⚠️ **Warning**</span>    
> If you want to execute the IGDB ingestion yourself, you must have and IGDB 
> developer key and an account.  
> Please follow the instructions in this link: https://api-docs.igdb.com/#getting-started.  
> Then write all your information in the `.env` file.

In [5]:
# We import this two helpers for ingestion
from ingest import IGDB, Wikipedia

In [ ]:
igdb = IGDB(opensearch_client)
# Uncomment the next line to ingest the IGDB data yourslef.
#igdb.download(index="igdb_2")

### <span style=color:deepskyblue>📄 Wikipedia ingestion</span>

**You can skip this section if you want.**

> For the wikipedia ingest, **you don't need to do any extra setup** (even if you 
> skipped igdb ingestion part).


We pull the wikipedia information from huggingface index. The data was updated 
on 


In [ ]:
wikipedia = Wikipedia(opensearch_client)
# Uncomment the next line to ingest the wikipedia data yourself.
#wikipedia.download(index="wikipedia_2")

## 🧪 <span style=color:forestgreen>Evaluation</span>

Here we will use our `Evaluator` class to execute each one of the steps:
1. Ground truth generation
2. Search evaluation & optimization.
    1. Perform the base evaluation itself.
    2. Optimize the boost values.
3. Tools use and final RAG's answer evaluation.

In [8]:
import pandas as pd
from evaluation import Evaluator

# Free llm model for the judge
evaluator = Evaluator(RAGClient(opensearch_client, model="gemma-4-31b-it"))

### 🎯 <span style=color:orangered>Ground truth generation</span>

In order to evaluate the search quality and model's performance we must have 
querys and a target variable to use as evaluation method.  
In this case, the target variable is the document id and the query will be a llm 
generated question based on the target document. 

For instance, if the target document is about Mario Kart, the llm
will generate `questions_per_doc` questions related to the document topic.

> <span style=color:yellow>⚠️ **Warning**</span>  
> The code blocks in this section create a very small dataset so as to
> not waste your tokens.  
> You may create a very large dataset if you want.  
> Keep in mind that there is a pre-generated large ground truth dataset, 
> so you don't need to create one from zero.

This should take $\sim 2\text{-}3 \ \text{min}$ 

In [9]:
rag_client.model = "gemma-4-31b-it"
igdb_ground_truth_small = evaluator.generate_ground_truth(
    index="igdb",
    questions_per_doc=3,
    num_docs=5,
    file_path="data/igdb_ground_truth_small.csv",
)

wikipedia_ground_truth_small = evaluator.generate_ground_truth(
    index="wikipedia",
    questions_per_doc=3,
    num_docs=5,
    file_path="data/wikipedia_ground_truth_small.csv",
)

rag_client.model = "gemini-3.1-flash-lite"


Generating GT:  20%|██        | 1/5 [00:04<00:17,  4.49s/it]

Schema violation: parsed output is None. Retrying...


Generating GT:  80%|████████  | 4/5 [00:14<00:03,  3.29s/it]

Schema violation: parsed output is None. Retrying...


Generating GT: 100%|██████████| 5/5 [00:17<00:00,  3.45s/it]


In [10]:
igdb_ground_truth_small = pd.read_csv("data/igdb_ground_truth_small.csv")
wikipedia_ground_truth_small = pd.read_csv("data/wikipedia_ground_truth_small.csv")

### 🔎 <span style=color:goldenrod>Search evaluation & optimization</span>

We must evaluate our search functions. Do they return the relevant documents?
For this we will use two metrics

If you use the full dataset this will take around $10$ min, you can bring a coffe ☕

In [11]:
num = 5  # Number of retrieved documents per search
igdb_score = {}
wikipedia_score = {}

evaluator.ground_truth = igdb_ground_truth_small
for search_type in ("lexical", "semantic", "hybrid"):
    time.sleep(0.2)
    hr_score, mrr_score, _ = evaluator.evaluate_search(
        index="igdb",
        search_type=search_type,
        num=num,
        max_workers=2,
    )

    igdb_score[search_type] = {"hr": hr_score, "mrr": mrr_score}

evaluator.ground_truth = wikipedia_ground_truth_small
for search_type in ("lexical", "semantic", "hybrid"):
    time.sleep(0.2)
    hr_score, mrr_score, _ = evaluator.evaluate_search(
        index="wikipedia",
        search_type=search_type,
        num=num,
        max_workers=2,
    )

    wikipedia_score[search_type] = {"hr": hr_score, "mrr": mrr_score}


Evaluating Search: 100%|██████████| 15/15 [00:01<00:00,  9.69it/s]


In [12]:
display(Markdown("# IGDB search score"))
display(pd.DataFrame(igdb_score))

display(Markdown("# Wikipedia search score"))
display(pd.DataFrame(wikipedia_score))

# IGDB search score

,lexical,semantic,hybrid
hr,0.933333,0.333333,0.933333
mrr,0.800000,0.250000,0.733333


# Wikipedia search score

,lexical,semantic,hybrid
hr,0.933333,0.133333,0.933333
mrr,0.866667,0.133333,0.833333


Now, let's test for the bigger datasets. This will take some time $\sim 10$ min 
depending on your hardware.

In [13]:
# Loads the larger pre-made ground truths.
igdb_ground_truth = pd.read_csv("data/igdb_ground_truth.csv")
wikipedia_ground_truth = pd.read_csv("data/wikipedia_ground_truth.csv")

In [14]:
output = input("Proceed with full search evaluation (This will take some time) [y/n]: ")

if output == 'y':
    num = 5  # Number of retrieved documents per search
    igdb_score = {}
    wikipedia_score = {}


    evaluator.ground_truth = igdb_ground_truth
    for search_type in ("lexical", "semantic", "hybrid"):
        hr_score, mrr_score, _ = evaluator.evaluate_search(
            index="igdb",
            search_type=search_type,
            num=num,
            max_workers=2,
        )

        igdb_score[search_type] = {"hr": hr_score, "mrr": mrr_score}

    evaluator.ground_truth = wikipedia_ground_truth
    for search_type in ("lexical", "semantic", "hybrid"):
        hr_score, mrr_score, _ = evaluator.evaluate_search(
            index="wikipedia",
            search_type=search_type,
            num=num,
            max_workers=2,
        )

        wikipedia_score[search_type] = {"hr": hr_score, "mrr": mrr_score}


Evaluating Search: 100%|██████████| 60/60 [00:06<00:00,  9.70it/s]


In [ ]:
display("IGDB search score")
display(pd.DataFrame(igdb_score).round(3))

display("Wikipedia search score")
display(pd.DataFrame(wikipedia_score).round(3))

'IGDB search score'

,lexical,semantic,hybrid
hr,0.843507,0.349612,0.803552
mrr,0.735313,0.258398,0.695727


'Wikipedia search score'

,lexical,semantic,hybrid
hr,0.800000,0.283333,0.783333
mrr,0.734722,0.213056,0.717500


Now we want to use a `boost_dict` which is a way to increase or decrease the 
importance/relevance of different search fields. For instance we may set a boost of $2$
for the *name* field and $3$ for the *storyline*, so that the search will tend to prioritize
word match more in the *storyline* field and less in the *name* field.

Here our optimization method is pure brute force, we will only test in a given set of values (n dimensional grid) and take the combination that returns the best result. It's simply that.

We will perform the optimization on each index (wikipedia and igdb) separately.
Given that this takes a long time we will just test a very small set of possible values. 

In [16]:
from itertools import product

#### IGDB Index search optimization

Here we will test two search methods the lexical search and hybrid search.
We want to tune the boosting parameters so as to have the maximum performance.  
This should take $\sim 3$ min

In [17]:
# performance_df = pd.DataFrame()
results = []

# Search boosting optimization
# For IGDB index

evaluator.ground_truth = igdb_ground_truth_small
vals = (0, 1, 2)
iter = 0

# Pre-calculate total iterations for the universal tqdm bar
total_iterations = len(
    list(product(vals, vals, vals, ("lexical", "semantic", "hybrid")))
)

from tqdm.auto import tqdm

with tqdm(total=total_iterations, desc="Optimizing IGDB Search") as pbar:
    for i, j, k, search_type in product(
        vals, vals, vals, ("lexical", "semantic", "hybrid")
    ):
        # Skip redundant boost combinations (i == j == k) where they are not the baseline (1)
        if (search_type != "semantic" and i == j == k != 1) or (
            search_type == "semantic" and (i != 0 or j != 0 or k != 0)
        ):
            pbar.update(1)
            continue

        # To avoid the Memory Circuit Breaker, we introduce a sleep
        # and limit concurrency in evaluate_search
        hr_score, mrr_score, _ = evaluator.evaluate_search(
            index="igdb",
            search_type=search_type,
            boost_dict={"name": i, "summary": j, "storyline": k},
            max_workers=2,  # Reduce concurrency to minimize memory pressure
        )

        results.append(
            {
                "name": i,
                "summary": j,
                "storyline": k,
                "hr": hr_score,
                "mrr": mrr_score,
                "type": search_type,
            }
        )

        
        time.sleep(0.2)
        iter = 0

        iter += 1
        pbar.update(1)

igdb_performance_df = pd.DataFrame(results)

Optimizing IGDB Search:   0%|          | 0/81 [00:00<?, ?it/s]

Evaluating Search: 100%|██████████| 15/15 [00:01<00:00,  8.82it/s]


In [18]:
df = igdb_performance_df.sort_values("mrr", ascending=False)
display(Markdown("## Lexical search result"))
display(df[df.type == "lexical"].head(5).round(3))

display(Markdown("## Semantic search result"))
display(df[df.type == "semantic"].head(5).round(3))

display(Markdown("## Hybrid search result"))
display(df[df.type == "hybrid"].head(5).round(3))

## Lexical search result

,name,summary,storyline,hr,mrr,type
25,1,1,1,0.933,0.800,lexical
33,1,2,2,0.933,0.767,lexical
7,0,1,1,0.867,0.733,lexical
15,0,2,2,0.867,0.733,lexical
49,2,2,1,0.800,0.689,lexical


## Semantic search result

,name,summary,storyline,hr,mrr,type
0,0,0,0,0.333,0.25,semantic


## Hybrid search result

,name,summary,storyline,hr,mrr,type
26,1,1,1,0.933,0.733,hybrid
34,1,2,2,0.933,0.683,hybrid
8,0,1,1,0.867,0.667,hybrid
16,0,2,2,0.867,0.667,hybrid
50,2,2,1,0.800,0.647,hybrid


#### Wikipedia index search optimization

In [19]:
evaluator.ground_truth = wikipedia_ground_truth_small

results = []
vals = (0, 1, 2, 3)
iter = 0

# Pre-calculate total iterations for the universal tqdm bar
total_iterations = len(list(product(vals, vals, ("lexical", "semantic", "hybrid"))))

from tqdm.auto import tqdm

with tqdm(total=total_iterations, desc="Optimizing Wikipedia Search") as pbar:
    for i, j, search_type in product(vals, vals, ("lexical", "semantic", "hybrid")):
        # Skip redundant boost combinations for non-lexical searches (semantic ignores boosts)
        # and skip symmetric combinations (i == j) where they are not the baseline (1)
        if (search_type != "semantic" and i == j != 1) or (
            search_type == "semantic" and (i != 0 or j != 0)
        ):
            pbar.update(1)
            continue

        # To avoid the Memory Circuit Breaker, we introduce a sleep
        # and limit concurrency in evaluate_search
        # time.sleep(0.2)
        hr_score, mrr_score, _ = evaluator.evaluate_search(
            index="wikipedia",
            search_type=search_type,
            boost_dict={"title": i, "text": j},
            max_workers=2,  # Reduce concurrency to minimize memory pressure
        )

        results.append(
            {
                "title": i,
                "text": j,
                "hr": hr_score,
                "mrr": mrr_score,
                "type": search_type,
            }
        )

        time.sleep(0.2)

        iter += 1
        pbar.update(1)

wikipedia_performance_df = pd.DataFrame(results)

Optimizing Wikipedia Search:   0%|          | 0/48 [00:00<?, ?it/s]

Evaluating Search: 100%|██████████| 15/15 [00:01<00:00,  9.14it/s]


In [20]:
df = wikipedia_performance_df.sort_values("mrr", ascending=False)
display(Markdown("## Lexical search"))
display(df[df.type == "lexical"].head(5).round(3))

display(Markdown("## Semantic search"))
display(df[df.type == "semantic"].head(5).round(3))

display(Markdown("## Hybrid search"))
display(df[df.type == "hybrid"].head(5).round(3))

## Lexical search

,title,text,hr,mrr,type
1,0,1,0.933,0.867,lexical
5,0,3,0.933,0.867,lexical
3,0,2,0.933,0.867,lexical
9,1,1,0.933,0.867,lexical
25,3,2,0.933,0.867,lexical


## Semantic search

,title,text,hr,mrr,type
0,0,0,0.133,0.133,semantic


## Hybrid search

,title,text,hr,mrr,type
6,0,3,0.933,0.833,hybrid
2,0,1,0.933,0.833,hybrid
20,2,3,0.933,0.833,hybrid
26,3,2,0.933,0.833,hybrid
14,1,3,0.933,0.833,hybrid


#### Final conclusion

Now that we tuned we can see which 

### 🔧 <span style=color:silver>Tools and final RAG answer evaluation</span>

Here is the final evaluation. We want to see the performance of our agent when
using the entire RAG system and the optimized boost values. So we use two 
sources of evaluation. The user's evaluation and a separate llm-judge evaluation. 
The judge will evaluate the tool usage and the final answer quality based on the 
ground truth. The user only evaluates the final answer with good or bad review.

> This will take some time ⏰. So we will only test with 5. This akes around $\sim 6$ min

In [27]:
judge = RAGClient(opensearch_client, model="gemma-4-31b-it")
rag_client.model = "gemma-4-31b-it"
evaluator.rag_client = rag_client
evaluator.ground_truth = igdb_ground_truth_small.iloc[:5,:]
evaluator.evaluate_agent(judge, index="igdb", max_workers=1)

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

Successfully retrieved the content.
Successfully retrieved the content.


Evaluating:  20%|██        | 1/5 [00:42<02:51, 42.91s/it]

Batch completed. Saving to database.
Successfully retrieved the content.
Quota exceeded (429). Retrying in 20.48 seconds...
Quota exceeded (429). Retrying in 40.82 seconds...
Successfully retrieved the content.


Evaluating:  40%|████      | 2/5 [04:22<07:20, 146.76s/it]

...

Batch completed. Saving to database.
Quota exceeded (429). Retrying in 20.11 seconds...
Quota exceeded (429). Retrying in 40.53 seconds...
Successfully retrieved the content.
Quota exceeded (429). Retrying in 20.53 seconds...
Quota exceeded (429). Retrying in 40.59 seconds...
Successfully retrieved the content.


Evaluating:  80%|████████  | 4/5 [10:56<03:05, 185.67s/it]

Batch completed. Saving to database.
Quota exceeded (429). Retrying in 20.36 seconds...
Quota exceeded (429). Retrying in 40.28 seconds...
Successfully retrieved the content.
Successfully retrieved the content.


Evaluating: 100%|██████████| 5/5 [12:52<00:00, 154.44s/it]

Batch completed. Saving to database.


In [28]:
from metrics import load_judge_feedback_data

judge_eval = load_judge_feedback_data()

def score_converter(x):
    match x:
        case "good":
            return 1
        case "average":
            return 0.5
        case "bad":
            return 0.0
        case _:
            return 0

score = judge_eval[["answer_score", "tool_score"]].map(score_converter)

print(score.sum()/len(score))


answer_score    0.2
tool_score      0.7
dtype: float64


In [ ]:
evaluator.ground_truth = wikipedia_ground_truth_small.iloc[:5,:]
evaluator.evaluate_agent(judge, index="wikipedia", max_workers=1)

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

Quota exceeded (429). Retrying in 20.31 seconds...
Successfully retrieved the content.
Quota exceeded (429). Retrying in 20.54 seconds...
Quota exceeded (429). Retrying in 40.01 seconds...
Successfully retrieved the content.


Evaluating:  20%|██        | 1/5 [01:57<07:49, 117.34s/it]

Batch completed. Saving to database.
Quota exceeded (429). Retrying in 20.27 seconds...
Successfully retrieved the content.
Quota exceeded (429). Retrying in 20.75 seconds...
Quota exceeded (429). Retrying in 40.64 seconds...
Successfully retrieved the content.


Evaluating:  40%|████      | 2/5 [03:48<05:41, 113.79s/it]

Batch completed. Saving to database.
Quota exceeded (429). Retrying in 20.85 seconds...
Quota exceeded (429). Retrying in 40.66 seconds...
Quota exceeded (429). Retrying in 80.55 seconds...
Quota exceeded (429). Retrying in 160.07 seconds...
Quota exceeded (429). Retrying in 320.82 seconds...


Evaluating:  60%|██████    | 3/5 [14:16<11:37, 348.73s/it]

Successfully retrieved the content.
Failed to process: Who is the main antagonist in ... Error: 'NoneType' object has no attribute 'candidates'
Quota exceeded (429). Retrying in 20.78 seconds...
Quota exceeded (429). Retrying in 40.62 seconds...
Quota exceeded (429). Retrying in 80.23 seconds...


## 📊 <span style=color:gold>Monitoring</span>

The final step is the monitoring, we must see the **usage** (tokens and cost) 
and **performance** (reviews) of our agent.  
For this, we can open the *streamlit* app in this address http://localhost:8501,
Click to the 

In [29]:
response, _ = rag_client.rag(
    "Tell me about Dark Souls. Give me information from both indices."
)
display(Markdown(response))

Successfully retrieved the content.
Quota exceeded (429). Retrying in 20.34 seconds...
Quota exceeded (429). Retrying in 40.98 seconds...
Successfully retrieved the content.


All attempts to use the IGDB index for "Dark Souls" have resulted in the same technical error. I have successfully retrieved information from the Wikipedia index. I will now synthesize the answer based on the Wikipedia evidence and explicitly state that I could not retrieve information from IGDB.
Based on the Wikipedia index, **Dark Souls** is a series of action role-playing games developed by FromSoftware and published by Bandai Namco Entertainment.

**Series Overview**
*   **Creation:** The series was created by Hidetaka Miyazaki.
*   **Titles:** The series began with *Dark Souls* (2011), followed by *Dark Souls II* (2014), and *Dark Souls III* (2016).
*   **Commercial Success:** By 2022, the series had shipped over 33 million copies.
*   **Legacy:** The first *Dark Souls* is frequently cited as one of the greatest games of all time. The series, along with other FromSoftware titles like *Demon's Souls*, *Bloodborne*, and *Elden Ring*, are commonly grouped together as "soulsbornes."

**Game Details**
*   **Dark Souls (2011):** A spiritual successor to *Demon's Souls*, set in the kingdom of Lordran. Players control a cursed undead character on a pilgrimage to discover the fate of their kind. It is praised for its combat depth and intricate level design, though its unforgiving difficulty is a point of both praise and criticism.
*   **Dark Souls III (2016):** The final entry in the series, played from a third-person perspective. It features various weapons, armor, and magic. This title saw the return of Hidetaka Miyazaki as director. It shipped over 10 million copies by 2020 and included two DLC expansions: *Ashes of Ariandel* and *The Ringed City*.

**Setting and Gameplay**
The games are set in a dark, medieval fantasy world where players battle knights, dragons, phantoms, demons, and other supernatural entities. The narrative and gameplay center around the accretion, loss, and recovery of "souls."

For the IGDB index, I couldn't retrieve the information due to a technical error.

In [30]:
print([usage.prompt_token_count for usage in rag_client.usage_history])
print([usage.candidates_token_count for usage in rag_client.usage_history])
print([usage.total_token_count for usage in rag_client.usage_history])

[1574, 2103]
[33, 423]
[1607, 2581]


Here is not very pleasent to see, open the address http://localhost:8501